# Step 2 — Per-cell feature tokens

Goal: turn each Cellpose-SAM predicted cell instance (Step 1's masks) into a feature
token `Z_{i,t} = [e_image, e_fluorescence, e_morphology, x, y, z, t]` that Step 3's
learned affinity model will consume. Design rationale in `20260906_PLAN.txt`.

**Scope**: `44b6` only for now (per project decision). `6bba`'s Cellpose-SAM segmentation
currently fails locally (0 precision/recall in Step 1) — fixing it is an explicit
out-of-scope follow-up, not blocking this notebook.

**`e_image` note**: Cellpose's `styles` output (the obvious candidate for a learned
image embedding) is verified dead in the installed cellpose>=4.0 — hardcoded to zeros
in `cellpose/core.py`/`cellpose/vit.py`, kept only "for backwards compatibility". So
`e_image` here is a handcrafted placeholder (per-instance intensity histogram), exactly
the fallback the original design anticipated. A real embedding is reachable by hooking
`model.net.encoder` (the actual SAM ViT backbone) — documented as a future upgrade, not
implemented here.

## 0. Setup

Reused verbatim from `01_cellpose_sam_segmentation.ipynb`: physical scale constants,
`DATA_ROOT`, `load_geff`, `predicted_centroids_um`, `gt_centroids_um`, `match_centroids`,
`precision_recall_f1`.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import zarr
from scipy.ndimage import center_of_mass
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.measure import regionprops_table, marching_cubes, mesh_surface_area

# physical scale used by the evaluation metric (confirmed in notebook/20260723_analysis.ipynb)
SCALE_Z, SCALE_Y, SCALE_X = 1.625, 0.40625, 0.40625
SCALE = np.array([SCALE_Z, SCALE_Y, SCALE_X])
VOXEL_VOLUME_UM3 = SCALE_Z * SCALE_Y * SCALE_X
MAX_MATCH_DIST_UM = 7.0

DATA_ROOT = "/mnt/d/KAGGLE/data/extracted/train"
SAMPLE_ID = "44b6_0db75fae"  # same sample used in notebook 01

# official test sample ids -- must never be used as a training example
TEST_IDS = {"44b6_0113de3b", "44b6_0b24845f", "6bba_05b6850b", "6bba_05db0fb1"}
assert SAMPLE_ID not in TEST_IDS

STEP1_CHECKPOINT_DIR = Path("_checkpoints/01_cellpose_sam")
CHECKPOINT_DIR = Path("_checkpoints/02_cell_tokens")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("zarr", zarr.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

zarr 3.1.6 | numpy 2.4.6 | pandas 3.0.5


In [2]:
def load_geff(sid):
    g = zarr.open_group(f"{DATA_ROOT}/{sid}.geff", mode="r")
    nodes = pd.DataFrame(
        {
            "node_id": g["nodes/ids"][:],
            "t": g["nodes/props/t/values"][:],
            "z": g["nodes/props/z/values"][:],
            "y": g["nodes/props/y/values"][:],
            "x": g["nodes/props/x/values"][:],
        }
    )
    edges = pd.DataFrame(g["edges/ids"][:], columns=["source", "target"])
    return nodes, edges


def available_ids(data_root, ids):
    """Filter an id list down to those with a .zarr present on disk."""
    data_root = Path(data_root)
    present = [sid for sid in ids if (data_root / f"{sid}.zarr").exists()]
    dropped = len(ids) - len(present)
    if dropped:
        print(f"available_ids: dropped {dropped}/{len(ids)} ids not present under {data_root}")
    return present


def predicted_centroids_um(masks, offset=(0, 0, 0)):
    labels = np.unique(masks)
    labels = labels[labels != 0]
    if len(labels) == 0:
        return np.empty((0, 3))
    centers = center_of_mass(masks, labels=masks, index=labels)
    centers = np.asarray(centers) + np.asarray(offset)
    return centers * SCALE


def gt_centroids_um(nodes_df, t):
    rows = nodes_df[nodes_df["t"] == t]
    return rows[["z", "y", "x"]].to_numpy(dtype=float) * SCALE


def match_centroids(pred_um, gt_um, max_dist=MAX_MATCH_DIST_UM):
    if len(pred_um) == 0 or len(gt_um) == 0:
        return [], list(range(len(pred_um))), list(range(len(gt_um)))
    cost = cdist(pred_um, gt_um)
    row_ind, col_ind = linear_sum_assignment(cost)
    matches = [(r, c) for r, c in zip(row_ind, col_ind) if cost[r, c] <= max_dist]
    matched_pred = {r for r, _ in matches}
    matched_gt = {c for _, c in matches}
    fp = [r for r in range(len(pred_um)) if r not in matched_pred]
    fn = [c for c in range(len(gt_um)) if c not in matched_gt]
    return matches, fp, fn


def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")
    return precision, recall, f1


nodes_df, edges_df = load_geff(SAMPLE_ID)
img_group = zarr.open_group(f"{DATA_ROOT}/{SAMPLE_ID}.zarr", mode="r")
arr = img_group["0"]
print("volume shape (T,Z,Y,X):", arr.shape, "dtype:", arr.dtype)
print("GT nodes:", len(nodes_df), "edges:", len(edges_df))

volume shape (T,Z,Y,X): (100, 64, 256, 256) dtype: uint16
GT nodes: 157 edges: 151


## 1. Load Step 1's checkpointed masks

The 3 existing `44b6` checkpoints (`t=0,29,58`) from Step 1. These are single, widely-spaced
frames -- fine for validating token-extraction *functions*, but not for Step 3's
frame-to-frame affinity training (that needs a contiguous window, built separately in
notebook 03 Section 0).

In [3]:
masks_by_t = {}
offsets_by_t = {}
intensity_by_t = {}

for ckpt_path in sorted(STEP1_CHECKPOINT_DIR.glob("44b6_t*.npz")):
    t = int(ckpt_path.stem.split("_t")[1])
    data = np.load(ckpt_path)
    masks = data["masks"]
    offset = tuple(int(v) for v in data["offset"])
    z0, y0, x0 = offset
    zs, ys, xs = masks.shape
    intensity_vol = np.asarray(arr[t, z0 : z0 + zs, y0 : y0 + ys, x0 : x0 + xs])

    masks_by_t[t] = masks
    offsets_by_t[t] = offset
    intensity_by_t[t] = intensity_vol
    n_instances = int(masks.max())
    print(f"t={t}: mask shape {masks.shape}, offset {offset}, {n_instances} instances")

assert masks_by_t, "no 44b6 checkpoints found -- run notebook 01 first"

t=0: mask shape (12, 48, 48), offset (13, 69, 0), 7 instances
t=29: mask shape (12, 48, 48), offset (18, 107, 26), 4 instances
t=58: mask shape (12, 48, 48), offset (32, 112, 119), 5 instances


## 2. `e_morphology` — shape features (physical units)

`skimage.measure.regionprops_table(..., spacing=SCALE)` gives volume/axis lengths/solidity
already in physical microns. 3D surface area has no direct regionprops property, so it's
computed per-instance via `marching_cubes` + `mesh_surface_area`. `elongation` and
`sphericity` are derived from these.

**Degenerate instances**: some masks from Step 1's 2D-per-plane + stitch approach are flat
along one axis (the stitch failed to link a detection across Z-slices), which makes an
ellipsoid/convex-hull fit meaningless (skimage silently returns absurd values like
`elongation=1e8` or `solidity=inf`). These are detected via bounding-box extent and their
shape-derived features are explicitly set to NaN and flagged in `is_flat_degenerate`,
rather than silently propagating garbage.

In [4]:
def compute_morphology_features(mask, spacing=SCALE):
    props = regionprops_table(
        mask, spacing=tuple(spacing),
        properties=("label", "area", "axis_major_length", "axis_minor_length", "solidity", "bbox"),
    )
    df = pd.DataFrame(props).rename(columns={"area": "volume_um3"})
    bbox_cols = [c for c in df.columns if c.startswith("bbox-")]
    extents = np.stack([df[f"bbox-{3 + i}"] - df[f"bbox-{i}"] for i in range(3)], axis=1)
    df["is_flat_degenerate"] = (extents == 1).any(axis=1)
    df = df.drop(columns=bbox_cols)

    surface_areas = []
    for label in df["label"]:
        submask = mask == label
        padded = np.pad(submask, 1)  # close the surface at crop edges
        try:
            verts, faces, _, _ = marching_cubes(padded.astype(np.float32), level=0.5, spacing=tuple(spacing))
            sa = mesh_surface_area(verts, faces)
        except (RuntimeError, ValueError):
            sa = np.nan
        surface_areas.append(sa)
    df["surface_area_um2"] = surface_areas
    df["elongation"] = df["axis_major_length"] / df["axis_minor_length"].replace(0, np.nan)
    df["sphericity"] = (np.pi ** (1 / 3)) * (6 * df["volume_um3"]) ** (2 / 3) / df["surface_area_um2"]

    shape_cols = ["elongation", "sphericity", "solidity"]
    df.loc[df["is_flat_degenerate"], shape_cols] = np.nan

    return df[["label", "volume_um3", "surface_area_um2", *shape_cols, "is_flat_degenerate"]]


_sample_morph = compute_morphology_features(masks_by_t[0])
_sample_morph

/home/sjm5701/anaconda3/envs/biohub/lib/python3.11/site-packages/skimage/measure/_regionprops.py:490: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6013 qhull input error: input is less than 3-dimensional since all points have the same x coordinate    0

While executing:  | qhull i Qt
Options selected for Qhull 2020.2.r 2020/08/31:
  run-id 485010540  incidence  Qtriangulate  _pre-merge  _zero-centrum
  _max-width  5  Error-roundoff 5.8e-15  _one-merge 4.1e-14  _near-inside 2e-13
  Visible-distance 1.2e-14  U-max-coplanar 1.2e-14  Width-outside 2.3e-14
  _wide-facet 7e-14  _maxoutside 4.7e-14

  return convex_hull_image(self.image)
/home/sjm5701/anaconda3/envs/biohub/lib/python3.11/site-packages/skimage/measure/_regionprops.py:704: RuntimeWarning: divide by zero encountered in scalar divide
  return self.area / self.area_convex
/home/sjm5701/anaconda3/envs/biohub/lib/python3.11/site-packages/skimage/measure/_regionprops.py:490: UserWarn

,label,volume_um3,surface_area_um2,elongation,sphericity,solidity,is_flat_degenerate
0,1,14.213989,35.851547,3.015371,0.791505,0.868852,False
1,2,251.828979,250.696097,1.750380,0.769261,0.822962,False
2,3,210.796143,217.628262,2.305781,0.787065,0.862788,False
3,4,186.927368,251.483435,3.392709,0.628671,0.601900,False
4,5,4.022827,12.289592,NaN,NaN,NaN,True
5,6,27.355225,56.052852,NaN,NaN,NaN,True
6,7,176.199829,203.025616,2.304798,0.748635,0.817164,False


## 3. `e_fluorescence` — intensity features

Single-channel intensity statistics under each mask (mean/std/percentiles/integrated
intensity) -- "fluorescence" per the original design naming, really just intensity features
since there's only one channel.

In [5]:
def compute_intensity_features(mask, intensity_vol):
    labels = np.unique(mask)
    labels = labels[labels != 0]
    rows = []
    for label in labels:
        vals = intensity_vol[mask == label].astype(np.float64)
        p10, p50, p90 = np.percentile(vals, [10, 50, 90])
        rows.append(
            {
                "label": int(label),
                "mean_intensity": vals.mean(),
                "std_intensity": vals.std(),
                "p10": p10, "p50": p50, "p90": p90,
                "integrated_intensity": vals.sum(),
            }
        )
    return pd.DataFrame(rows)


compute_intensity_features(masks_by_t[0], intensity_by_t[0])

,label,mean_intensity,std_intensity,p10,p50,p90,integrated_intensity
0,1,699.603774,76.986728,602.4,720.0,786.8,37079.0
1,2,1125.821086,237.162643,796.0,1118.0,1459.2,1057146.0
2,3,1102.422392,246.571075,753.5,1111.0,1428.0,866504.0
3,4,854.929699,124.951891,698.0,840.0,1029.0,595886.0
4,5,676.733333,46.086103,629.2,673.0,739.2,10151.0
5,6,1146.303922,156.636509,934.9,1134.5,1353.8,116923.0
6,7,1148.718417,204.112643,878.6,1163.0,1409.8,754708.0


## 4. Position / time features

Both a physical-µm centroid (for distance computations, reusing `predicted_centroids_um`)
and a raw-voxel centroid (the submission CSV needs integer voxel coordinates).

In [6]:
def compute_voxel_centroids(mask, offset=(0, 0, 0)):
    labels = np.unique(mask)
    labels = labels[labels != 0]
    if len(labels) == 0:
        return pd.DataFrame(columns=["label", "z_vox", "y_vox", "x_vox"])
    centers = center_of_mass(mask, labels=mask, index=labels)
    centers = np.asarray(centers) + np.asarray(offset)
    df = pd.DataFrame(centers, columns=["z_vox", "y_vox", "x_vox"])
    df.insert(0, "label", labels)
    df["z_um"] = df["z_vox"] * SCALE_Z
    df["y_um"] = df["y_vox"] * SCALE_Y
    df["x_um"] = df["x_vox"] * SCALE_X
    return df


compute_voxel_centroids(masks_by_t[0], offsets_by_t[0])

,label,z_vox,y_vox,x_vox,z_um,y_um,x_um
0,1,13.471698,76.245283,45.169811,21.891509,30.974646,18.350236
1,2,14.851970,91.439830,33.093717,24.134452,37.147431,13.444322
2,3,15.647583,102.174300,6.169211,25.427322,41.508309,2.506242
3,4,15.578192,106.733142,26.205165,25.314562,43.360339,10.645848
4,5,14.000000,111.600000,45.066667,22.750000,45.337500,18.308333
5,6,18.000000,94.686275,43.029412,29.250000,38.466299,17.480699
6,7,22.216134,81.272451,30.846271,36.101218,33.016933,12.531298


## 5. `e_image` — placeholder image feature

Per-instance normalized intensity histogram. As noted above, Cellpose's `styles` output is
verified dead (hardcoded zeros), so this handcrafted placeholder is the MVP `e_image` per
the original design's own documented fallback. A real SAM-ViT-encoder embedding
(`model.net.encoder`) is a scoped future upgrade, not built here.

In [7]:
def compute_image_placeholder_features(mask, intensity_vol, n_bins=16, hist_range=None):
    labels = np.unique(mask)
    labels = labels[labels != 0]
    if hist_range is None:
        hist_range = (float(intensity_vol.min()), float(intensity_vol.max()))
    rows = []
    for label in labels:
        vals = intensity_vol[mask == label].astype(np.float64)
        hist, _ = np.histogram(vals, bins=n_bins, range=hist_range, density=True)
        row = {"label": int(label)}
        row.update({f"hist_{i}": h for i, h in enumerate(hist)})
        rows.append(row)
    return pd.DataFrame(rows)


compute_image_placeholder_features(masks_by_t[0], intensity_by_t[0]).iloc[:, :6]

,label,hist_0,hist_1,hist_2,hist_3,hist_4
0,1,0.0,0.0,0.0,0.0,0.000784
1,2,0.0,0.0,0.0,0.0,0.000000
2,3,0.0,0.0,0.0,0.0,0.000026
3,4,0.0,0.0,0.0,0.0,0.000000
4,5,0.0,0.0,0.0,0.0,0.000000
5,6,0.0,0.0,0.0,0.0,0.000000
6,7,0.0,0.0,0.0,0.0,0.000000


## 6. Assemble the per-cell token table

Joins morphology + intensity + position/time + placeholder-image on `(t, label)` into one
wide DataFrame -- this is `Z_{i,t}` as a table row.

In [8]:
def build_token_table(masks_by_t, offsets_by_t, intensity_by_t, sample_id):
    frames = []
    for t, mask in masks_by_t.items():
        offset = offsets_by_t[t]
        intensity_vol = intensity_by_t[t]
        morph = compute_morphology_features(mask)
        inten = compute_intensity_features(mask, intensity_vol)
        pos = compute_voxel_centroids(mask, offset)
        img_feat = compute_image_placeholder_features(mask, intensity_vol)
        token = morph.merge(inten, on="label").merge(pos, on="label").merge(img_feat, on="label")
        token.insert(0, "t", t)
        frames.append(token)
    table = pd.concat(frames, ignore_index=True)
    table.insert(0, "sample_id", sample_id)
    return table


token_table = build_token_table(masks_by_t, offsets_by_t, intensity_by_t, SAMPLE_ID)
print(token_table.shape)
token_table.head()

(16, 37)


/home/sjm5701/anaconda3/envs/biohub/lib/python3.11/site-packages/skimage/measure/_regionprops.py:490: UserWarning: Failed to get convex hull image. Returning empty image, see error message below:
QH6013 qhull input error: input is less than 3-dimensional since all points have the same x coordinate    0

While executing:  | qhull i Qt
Options selected for Qhull 2020.2.r 2020/08/31:
  run-id 485010540  incidence  Qtriangulate  _pre-merge  _zero-centrum
  _max-width  5  Error-roundoff 5.8e-15  _one-merge 4.1e-14  _near-inside 2e-13
  Visible-distance 1.2e-14  U-max-coplanar 1.2e-14  Width-outside 2.3e-14
  _wide-facet 7e-14  _maxoutside 4.7e-14

  return convex_hull_image(self.image)
/home/sjm5701/anaconda3/envs/biohub/lib/python3.11/site-packages/skimage/measure/_regionprops.py:704: RuntimeWarning: divide by zero encountered in scalar divide
  return self.area / self.area_convex
/home/sjm5701/anaconda3/envs/biohub/lib/python3.11/site-packages/skimage/measure/_regionprops.py:490: UserWarn

,sample_id,t,label,volume_um3,surface_area_um2,elongation,sphericity,solidity,is_flat_degenerate,mean_intensity,...,hist_6,hist_7,hist_8,hist_9,hist_10,hist_11,hist_12,hist_13,hist_14,hist_15
0,44b6_0db75fae,0,1,14.213989,35.851547,3.015371,0.791505,0.868852,False,699.603774,...,0.004901,0.001372,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,44b6_0db75fae,0,2,251.828979,250.696097,1.750380,0.769261,0.822962,False,1125.821086,...,0.000741,0.000985,0.000940,0.001649,0.001494,0.001261,0.001239,0.000863,0.000719,0.000343
2,44b6_0db75fae,0,3,210.796143,217.628262,2.305781,0.787065,0.862788,False,1102.422392,...,0.000925,0.001005,0.001137,0.001322,0.001388,0.001176,0.001322,0.000978,0.000489,0.000304
3,44b6_0db75fae,0,4,186.927368,251.483435,3.392709,0.628671,0.601900,False,854.929699,...,0.002460,0.002907,0.002296,0.001312,0.000596,0.000015,0.000000,0.000000,0.000000,0.000000
4,44b6_0db75fae,0,5,4.022827,12.289592,NaN,NaN,NaN,True,676.733333,...,0.003463,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


## 7. Verification / success criteria

- No NaN/inf in non-degenerate rows' shape features.
- Schema + summary stats reviewed for sane physical ranges.
- Manual volume spot-check against `regionprops_table`'s `spacing=` output.
- Save to parquet and confirm round-trip reload.

In [9]:
print("columns:", list(token_table.columns))
print("n_flat_degenerate:", int(token_table["is_flat_degenerate"].sum()), "/", len(token_table))

non_deg = token_table[~token_table["is_flat_degenerate"]]
shape_cols = ["elongation", "sphericity", "solidity"]
assert not non_deg[shape_cols].isin([np.inf, -np.inf]).any().any(), "unexpected inf in non-degenerate rows"
assert not non_deg[shape_cols].isna().any().any(), "unexpected NaN in non-degenerate rows"
assert non_deg["sphericity"].between(0, 1).all(), "sphericity out of (0,1] range"

token_table[["t", "label", "volume_um3", "surface_area_um2", *shape_cols, "mean_intensity", "is_flat_degenerate"]].describe(include="all")

columns: ['sample_id', 't', 'label', 'volume_um3', 'surface_area_um2', 'elongation', 'sphericity', 'solidity', 'is_flat_degenerate', 'mean_intensity', 'std_intensity', 'p10', 'p50', 'p90', 'integrated_intensity', 'z_vox', 'y_vox', 'x_vox', 'z_um', 'y_um', 'x_um', 'hist_0', 'hist_1', 'hist_2', 'hist_3', 'hist_4', 'hist_5', 'hist_6', 'hist_7', 'hist_8', 'hist_9', 'hist_10', 'hist_11', 'hist_12', 'hist_13', 'hist_14', 'hist_15']
n_flat_degenerate: 4 / 16


,t,label,volume_um3,surface_area_um2,elongation,sphericity,solidity,mean_intensity,is_flat_degenerate
count,16.000000,16.000000,16.000000,16.000000,12.000000,12.000000,12.000000,16.000000,16
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12
mean,25.375000,3.312500,79.836357,105.169304,2.129755,0.802052,0.860458,1191.433385,NaN
std,25.666775,1.815443,79.232379,78.874468,0.596077,0.066391,0.089559,328.830071,NaN
min,0.000000,1.000000,4.022827,12.289592,1.303457,0.628671,0.601900,676.733333,NaN
25%,0.000000,2.000000,26.952942,55.042300,1.743639,0.782614,0.852832,1092.592836,NaN
50%,29.000000,3.000000,46.798889,76.102314,1.961554,0.815292,0.887179,1136.062504,NaN
75%,58.000000,4.250000,111.231171,135.733381,2.329225,0.849577,0.903831,1253.084080,NaN


In [10]:
# manual volume spot-check: recompute one instance's volume outside regionprops_table
spot_t, spot_label = 0, int(token_table[token_table["t"] == 0]["label"].iloc[0])
manual_vol = np.count_nonzero(masks_by_t[spot_t] == spot_label) * VOXEL_VOLUME_UM3
table_vol = token_table.loc[
    (token_table["t"] == spot_t) & (token_table["label"] == spot_label), "volume_um3"
].iloc[0]
print(f"manual volume: {manual_vol:.4f} um3 | table volume: {table_vol:.4f} um3")
assert abs(manual_vol - table_vol) < 1e-6
print("volume spot-check OK")

manual volume: 14.2140 um3 | table volume: 14.2140 um3
volume spot-check OK


In [11]:
out_path = CHECKPOINT_DIR / f"{SAMPLE_ID}.parquet"
token_table.to_parquet(out_path, index=False)

reloaded = pd.read_parquet(out_path)
pd.testing.assert_frame_equal(token_table.reset_index(drop=True), reloaded.reset_index(drop=True))
print(f"saved + round-trip verified: {out_path} ({len(token_table)} rows)")

saved + round-trip verified: _checkpoints/02_cell_tokens/44b6_0db75fae.parquet (16 rows)


## Next steps

- Proceed to `03_tracking_solver.ipynb` (Step 3): candidate edges, learned affinity, graph
  solver, and the actual competition-metric score.
- Known limitations carried forward: `44b6`-only scope, placeholder `e_image`, single-sample
  token table (only the 3 Step-1 checkpoint frames).